In [7]:
import csv
import pickle

In [8]:
class CurrencyAmount:
    def __init__(self, amount: float, currency: str):
        self.amount = amount
        self.currency = currency.upper()  

    def to_byn(self, rates: dict) -> float:
        """Конвертирует сумму в BYN по курсу из словаря"""
        if self.currency not in rates:
            raise ValueError(f"Неизвестная валюта: {self.currency}")
        return self.amount * rates[self.currency]

    def __repr__(self):
        return f"{self.amount} {self.currency}"

In [9]:
exchange_rates = {
    'USD': 3.27,
    'EUR': 3.55,
    'GBP': 4.15,
    'RUB': 0.035,
    'CHF': 3.60
}

In [10]:
def read_csv_transactions(filename: str) -> list[CurrencyAmount]:
    """
    Читает CSV-файл и возвращает список объектов CurrencyAmount
    Формат CSV: amount,currency
    """
    transactions = []
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            amount = float(row['amount'])
            currency = row['currency'].strip()
            transactions.append(CurrencyAmount(amount, currency))
    return transactions

In [11]:
def convert_and_save_to_binary(transactions: list[CurrencyAmount], rates: dict, output_file: str):
    """
    Конвертирует все суммы в BYN и записывает результат в бинарный файл.
    Результат: список словарей с исходной суммой, валютой и BYN-суммой
    """
    results = []

    for tx in transactions:
        try:
            byn_amount = tx.to_byn(rates)
            results.append({
                'original_amount': tx.amount,
                'currency': tx.currency,
                'byn_amount': round(byn_amount, 2)  # округляем до 2 знаков
            })
        except ValueError as e:
            print(f"⚠️ Пропущена транзакция: {tx} — {e}")
            continue

    # Записываем в бинарный файл
    with open(output_file, 'wb') as f:
        pickle.dump(results, f)

    print(f"✅ Успешно записано {len(results)} записей в {output_file}")
    return results

In [12]:
def main():
    # 1. Курсы валют (словарь)
    exchange_rates = {
        'USD': 3.27,
        'EUR': 3.55,
        'GBP': 4.15,
        'RUB': 0.035,
        'CHF': 3.60
    }

    print("📚 Чтение данных из input.csv...")
    transactions = read_csv_transactions('input.csv')

    print("📊 Прочитанные транзакции:")
    for tx in transactions:
        print(f"  - {tx}")

    print("\n🔄 Конвертация в BYN и запись в output.bin...")
    results = convert_and_save_to_binary(transactions, exchange_rates, 'output.bin')

    print("\n🔍 Проверка: считывание из output.bin...")
    with open('output.bin', 'rb') as f:
        loaded_results = pickle.load(f)

    print("Результаты из бинарного файла:")
    for res in loaded_results:
        print(f"  - {res['original_amount']} {res['currency']} = {res['byn_amount']} BYN")

if __name__ == "__main__":
    main()

📚 Чтение данных из input.csv...
📊 Прочитанные транзакции:
  - 100.0 USD
  - 250.0 EUR
  - 75.0 GBP
  - 500.0 RUB
  - 30.0 CHF

🔄 Конвертация в BYN и запись в output.bin...
✅ Успешно записано 5 записей в output.bin

🔍 Проверка: считывание из output.bin...
Результаты из бинарного файла:
  - 100.0 USD = 327.0 BYN
  - 250.0 EUR = 887.5 BYN
  - 75.0 GBP = 311.25 BYN
  - 500.0 RUB = 17.5 BYN
  - 30.0 CHF = 108.0 BYN
